## NLP Preprocessing and Text Classification


Setup:

In [32]:
import pandas as pd
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# Download necessary NLTK data (run once)
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

# Load Data from train.csv and test.csv
try:
    train_df = pd.read_csv('/content/sample_data/train.csv', encoding='latin1')
    test_df = pd.read_csv('/content/sample_data/test.csv', encoding='latin1')

    # Select only 'text' and 'sentiment' columns
    train_df = train_df[['text', 'sentiment']]
    test_df = test_df[['text', 'sentiment']]

    # Filter for 'positive' and 'negative' sentiments, excluding 'neutral' if present
    train_df = train_df[train_df['sentiment'].isin(['positive', 'negative','neutral'])]
    test_df = test_df[test_df['sentiment'].isin(['positive', 'negative','neutral'])]

    print("Train Data Head:")
    display(train_df.head())
    print("\nTest Data Head:")
    display(test_df.head())
except FileNotFoundError:
    print("Error: train.csv or test.csv not found. Please ensure they are in /content/sample_data/")
    train_df = pd.DataFrame(data)
    test_df = pd.DataFrame(data).sample(frac=0.3, random_state=42) # Create a small test set from sample data
    print("Using fallback sample data.")
    display(train_df)

Train Data Head:


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


,text,sentiment
0,"I`d have responded, if I were going",neutral
1,Sooo SAD I will miss you here in San Diego!!!,negative
2,my boss is bullying me...,negative
3,what interview! leave me alone,negative
4,"Sons of ****, why couldn`t they put them on t...",negative



Test Data Head:


,text,sentiment
0,Last session of the day http://twitpic.com/67ezh,neutral
1,Shanghai is also really exciting (precisely -...,positive
2,"Recession hit Veronique Branquinho, she has to...",negative
3,happy bday!,positive
4,http://twitpic.com/4w75p - I like it!!,positive


### 2. NLP Preprocessing

Preprocessing text is crucial to clean and standardize the data before feeding it to a machine learning model. We will cover tokenization, stopword removal, and lemmatization.

In [36]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    # Ensure text is a string to prevent AttributeError for non-string types (e.g., floats/NaN)
    if not isinstance(text, str):
        text = str(text) # Convert non-string types to string

    # Tokenization: Breaking text into individual words.
    tokens = word_tokenize(text.lower()) # Convert to lowercase and tokenize

    # Stopword Removal: Removing common words that don't add much meaning.
    filtered_tokens = [word for word in tokens if word.isalnum() and word not in stop_words]

    # Lemmatization: Reducing words to their base or root form.
    lemmas = [lemmatizer.lemmatize(word) for word in filtered_tokens]

    return " ".join(lemmas)

train_df['processed_text'] = train_df['text'].apply(preprocess_text)
test_df['processed_text'] = test_df['text'].apply(preprocess_text)

print("Processed Train Data Head:")
display(train_df[['text', 'processed_text']].head())
print("\nProcessed Test Data Head:")
display(test_df[['text', 'processed_text']].head())

Processed Train Data Head:


,text,processed_text
0,"I`d have responded, if I were going",responded going
1,Sooo SAD I will miss you here in San Diego!!!,sooo sad miss san diego
2,my boss is bullying me...,bos bullying
3,what interview! leave me alone,interview leave alone
4,"Sons of ****, why couldn`t they put them on t...",son put release already bought



Processed Test Data Head:


,text,processed_text
0,Last session of the day http://twitpic.com/67ezh,last session day http
1,Shanghai is also really exciting (precisely -...,shanghai also really exciting precisely skyscr...
2,"Recession hit Veronique Branquinho, she has to...",recession hit veronique branquinho quit compan...
3,happy bday!,happy bday
4,http://twitpic.com/4w75p - I like it!!,http like


### 3. Text Vectorization

Machine learning models cannot directly understand text. Text vectorization converts text into numerical representations (vectors). We will demonstrate `CountVectorizer` and `TF-IDF`.

#### a. CountVectorizer

`CountVectorizer` converts a collection of text documents to a matrix of token counts. It simply counts the occurrences of each word.

In [37]:
count_vectorizer = CountVectorizer()
X_count_train = count_vectorizer.fit_transform(train_df['processed_text'])
X_count_test = count_vectorizer.transform(test_df['processed_text'])

print("CountVectorizer Feature Names:")
print(count_vectorizer.get_feature_names_out())
print("\nCountVectorizer Train Output (first 2 rows, dense representation for display):")
display(pd.DataFrame(X_count_train.toarray(), columns=count_vectorizer.get_feature_names_out()).head(2))
print("\nCountVectorizer Test Output (first 2 rows, dense representation for display):")
display(pd.DataFrame(X_count_test.toarray(), columns=count_vectorizer.get_feature_names_out()).head(2))

CountVectorizer Feature Names:
['00' '000' '04' ... 'zzzzy' 'zzzzzzz' 'zzzzzzzzzzzzzzz']

CountVectorizer Train Output (first 2 rows, dense representation for display):


,00,000,04,05,06,0600,07,08,09,0f,...,zum,zumba,zune,zwarte,zx33t1,zyrtec,zzzz,zzzzy,zzzzzzz,zzzzzzzzzzzzzzz
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0



CountVectorizer Test Output (first 2 rows, dense representation for display):


,00,000,04,05,06,0600,07,08,09,0f,...,zum,zumba,zune,zwarte,zx33t1,zyrtec,zzzz,zzzzy,zzzzzzz,zzzzzzzzzzzzzzz
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


#### b. TF-IDF Vectorizer

TF-IDF (Term Frequency-Inverse Document Frequency) reflects how important a word is to a document in a collection or corpus. It increases with the number of times a word appears in the document but is offset by the frequency of the word in the corpus.

In [38]:
tfidf_vectorizer = TfidfVectorizer()
X_tfidf_train = tfidf_vectorizer.fit_transform(train_df['processed_text'])
X_tfidf_test = tfidf_vectorizer.transform(test_df['processed_text'])

print("TF-IDF Feature Names:")
print(tfidf_vectorizer.get_feature_names_out())
print("\nTF-IDF Vectorizer Train Output (first 2 rows, dense representation for display):")
display(pd.DataFrame(X_tfidf_train.toarray(), columns=tfidf_vectorizer.get_feature_names_out()).head(2))
print("\nTF-IDF Vectorizer Test Output (first 2 rows, dense representation for display):")
display(pd.DataFrame(X_tfidf_test.toarray(), columns=tfidf_vectorizer.get_feature_names_out()).head(2))

TF-IDF Feature Names:
['00' '000' '04' ... 'zzzzy' 'zzzzzzz' 'zzzzzzzzzzzzzzz']

TF-IDF Vectorizer Train Output (first 2 rows, dense representation for display):


,00,000,04,05,06,0600,07,08,09,0f,...,zum,zumba,zune,zwarte,zx33t1,zyrtec,zzzz,zzzzy,zzzzzzz,zzzzzzzzzzzzzzz
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0



TF-IDF Vectorizer Test Output (first 2 rows, dense representation for display):


,00,000,04,05,06,0600,07,08,09,0f,...,zum,zumba,zune,zwarte,zx33t1,zyrtec,zzzz,zzzzy,zzzzzzz,zzzzzzzzzzzzzzz
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


### 4. Classification Model

 The TF-IDF vectorized data to train a simple Logistic Regression model, a common choice for text classification.

In [39]:
# Using TF-IDF vectorized data for model training
X_train = X_tfidf_train
X_test = X_tfidf_test
y_train = train_df['sentiment']
y_test = test_df['sentiment']

print(f"Training set size: {X_train.shape[0]} samples")
print(f"Test set size: {X_test.shape[0]} samples")

# Initialize and train a Logistic Regression model
model = LogisticRegression(random_state=42)
model.fit(X_train, y_train)

print("\nModel trained successfully!")

Training set size: 27481 samples
Test set size: 3534 samples

Model trained successfully!


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


### 5. Evaluation metrcics


In [41]:
y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print(f"Model Accuracy: {accuracy:.2f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Model Accuracy: 0.70

Classification Report:
              precision    recall  f1-score   support

    negative       0.72      0.64      0.68      1001
     neutral       0.64      0.75      0.69      1430
    positive       0.81      0.71      0.75      1103

    accuracy                           0.70      3534
   macro avg       0.72      0.70      0.71      3534
weighted avg       0.71      0.70      0.71      3534

